In [1]:
import redis
import hashlib
import json
import time
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv(override=True)

# ─────────────────────────────────────────────
#  REDIS CONNECTION
# ─────────────────────────────────────────────
r = redis.Redis(
    host=os.getenv('REDIS_HOST', 'localhost'),
    port=int(os.getenv('REDIS_PORT', 6379)),
    decode_responses=True
)

# ─────────────────────────────────────────────
#  LLM INITIALIZATION (via Evren API)
# ─────────────────────────────────────────────
llm = ChatOpenAI(
    model=os.getenv("EVREN_MODEL"),
    api_key=os.getenv("EVREN_API_KEY"),
    base_url=os.getenv("EVREN_BASE_URL"),
    temperature=0
)

# ─────────────────────────────────────────────
#  HEALTH CHECK
# ─────────────────────────────────────────────
r.ping()

print("\n" + "═" * 58)
print("\033[1;36m  🚀  SYSTEM INITIALIZATION\033[0m")
print("═" * 58)

redis_status = "\033[1;32m✅ CONNECTED\033[0m" if r.ping() else "\033[1;31m❌ FAILED\033[0m"
print(f"  \033[1;33m▸ Redis\033[0m            → {redis_status}")
print(f"     \033[90mhost:\033[0m {os.getenv('REDIS_HOST', 'localhost')}:{os.getenv('REDIS_PORT', 6379)}")

api_status = "\033[1;32m✅ LOADED\033[0m" if os.getenv("EVREN_API_KEY") else "\033[1;31m❌ MISSING\033[0m"
print(f"  \033[1;33m▸ Evren API Key\033[0m    → {api_status}")

model_name = os.getenv("EVREN_MODEL", "unknown")
print(f"  \033[1;33m▸ LLM Model\033[0m        → \033[1;32m{model_name}\033[0m")
print(f"     \033[90mbase_url:\033[0m {os.getenv('EVREN_BASE_URL', 'N/A')}")

print("═" * 58 + "\n")


══════════════════════════════════════════════════════════
  🚀  SYSTEM INITIALIZATION
══════════════════════════════════════════════════════════
  ▸ Redis            → ✅ CONNECTED
     host: localhost:6379
  ▸ Evren API Key    → ✅ LOADED
  ▸ LLM Model        → deepseek-v4-flash
     base_url: https://evren-llmapi.ssyz.org.tr/v1
══════════════════════════════════════════════════════════



In [2]:

# Fictional company knowledge base for RAG testing.
# In production, these facts would come from PDFs,
# websites, databases, or other data sources.

KNOWLEDGE_BASE = [
    "NovaTech was founded in 2020 by Emily Carter and Daniel Brooks in Austin, Texas.",
    "NovaTech's core product is an AI-powered data analytics platform called InsightFlow.",
    "InsightFlow offers three subscription tiers: Starter ($39/month), Professional ($119/month), and Enterprise (custom pricing).",
    "NovaTech has over 3,200 customers across 35 countries as of 2025.",
    "The Starter plan includes up to 10 users, 20GB storage, and basic reporting features.",
    "The Professional plan includes up to 75 users, 200GB storage, advanced analytics, and API access.",
    "Enterprise customers get unlimited users, dedicated infrastructure, SLA guarantees, and a dedicated support team.",
    "NovaTech's refund policy: Starter and Professional plans have a 21-day money-back guarantee. Enterprise contracts are non-refundable after 45 days.",
    "NovaTech's customer support operates 24/7 for Enterprise customers and 8am-8pm CST Monday-Saturday for other plans.",
    "InsightFlow integrates with Salesforce, HubSpot, Slack, Microsoft Teams, and over 60 other tools via native connectors.",
    "NovaTech raised $60 million in Series B funding in June 2024, led by Sequoia Capital.",
    "NovaTech's data is stored in AWS us-west-2 and eu-central-1 regions, with optional on-premise deployment for Enterprise.",
    "All data is encrypted at rest using AES-256 and in transit using TLS 1.3.",
    "NovaTech is SOC 2 Type II certified and GDPR compliant as of March 2025.",
    "The InsightFlow mobile app is available on iOS (15+) and Android (11+).",
]

print(f"Knowledge base loaded: {len(KNOWLEDGE_BASE)} documents")

for i, doc in enumerate(KNOWLEDGE_BASE[:3], 1):
    print(f"  Doc {i}: {doc[:90]}...")

print("  ...")


Knowledge base loaded: 15 documents
  Doc 1: NovaTech was founded in 2020 by Emily Carter and Daniel Brooks in Austin, Texas....
  Doc 2: NovaTech's core product is an AI-powered data analytics platform called InsightFlow....
  Doc 3: InsightFlow offers three subscription tiers: Starter ($39/month), Professional ($119/month...
  ...


In [3]:
from sentence_transformers import SentenceTransformer


embedding_model = SentenceTransformer("all-MiniLM-L6-v2")


class EmbeddingCache:
    """Cache embeddings in Redis. Same text = instant retrieval."""

    def __init__(self, redis_client, embed_model=None, ttl_seconds: int = None):
        self.r = redis_client
        self.model = embed_model or embedding_model
        self.ttl = ttl_seconds  # None = no expiry (embeddings don't change)
        self._hits = 0
        self._misses = 0

    def _key(self, text: str) -> str:
        
        return f"embedding:minilm:{hashlib.sha256(text.encode()).hexdigest()[:16]}"

    def get_or_compute(self, text: str) -> list:
        """Return cached embedding or compute + store a new one."""
        key = self._key(text)
        cached = self.r.get(key)

        if cached:
            self._hits += 1
            return json.loads(cached)

        # Compute locally via HuggingFace / SentenceTransformer (no API cost/quota)
        self._misses += 1
        embedding = self.model.encode(text.strip()).tolist()

        # Store in Redis
        if self.ttl:
            self.r.set(key, json.dumps(embedding), ex=self.ttl)
        else:
            self.r.set(key, json.dumps(embedding))

        return embedding

    @property
    def stats(self) -> dict:
        total = self._hits + self._misses
        return {
            "hits": self._hits,
            "misses": self._misses,
            "hit_rate": f"{self._hits/total*100:.0f}%" if total > 0 else "N/A",
            "keys_in_redis": len(self.r.keys("embedding:minilm:*"))
        }


# ─────────────────────────────────────────────────
# 2. CACHE & INDEXING TEST
# ─────────────────────────────────────────────────
embed_cache = EmbeddingCache(r, embed_model=embedding_model)

for k in r.keys("embedding:minilm:*"):
    r.delete(k)

print("Indexing knowledge base (computing + caching embeddings via all-MiniLM-L6-v2)...")
start = time.time()
doc_embeddings = {}

for i, doc in enumerate(KNOWLEDGE_BASE):
    doc_embeddings[doc] = embed_cache.get_or_compute(doc)
    print(f"  [{i+1:>2}/{len(KNOWLEDGE_BASE)}] Indexed: {doc[:70]}...")

elapsed = time.time() - start
print(f"\nFirst pass: {elapsed:.2f}s (computed locally with all-MiniLM-L6-v2, stored in Redis)")
print(f"Cache stats: {embed_cache.stats}")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Indexing knowledge base (computing + caching embeddings via all-MiniLM-L6-v2)...
  [ 1/15] Indexed: NovaTech was founded in 2020 by Emily Carter and Daniel Brooks in Aust...
  [ 2/15] Indexed: NovaTech's core product is an AI-powered data analytics platform calle...
  [ 3/15] Indexed: InsightFlow offers three subscription tiers: Starter ($39/month), Prof...
  [ 4/15] Indexed: NovaTech has over 3,200 customers across 35 countries as of 2025....
  [ 5/15] Indexed: The Starter plan includes up to 10 users, 20GB storage, and basic repo...
  [ 6/15] Indexed: The Professional plan includes up to 75 users, 200GB storage, advanced...
  [ 7/15] Indexed: Enterprise customers get unlimited users, dedicated infrastructure, SL...
  [ 8/15] Indexed: NovaTech's refund policy: Starter and Professional plans have a 21-day...
  [ 9/15] Indexed: NovaTech's customer support operates 24/7 for Enterprise customers and...
  [10/15] Indexed: InsightFlow integrates with Salesforce, HubSpot, Slack, Microsoft Te

In [4]:
# Second pass — ALL embeddings come from Redis now
print("\n" + "═" * 68)
print("\033[1;36m  ⚡  SECOND PASS — CACHE-ONLY\033[0m")
print("\033[90m  All embeddings should come from Redis (no API calls)\033[0m")
print("═" * 68)

start = time.time()
for doc in KNOWLEDGE_BASE:
    _ = embed_cache.get_or_compute(doc)
elapsed = time.time() - start

print(f"\n  \033[1;34m▸ Total time\033[0m     → \033[1;32m{elapsed * 1000:.0f} ms\033[0m")
print(f"  \033[1;34m▸ Documents\033[0m      → \033[1;37m{len(KNOWLEDGE_BASE)}\033[0m")

# ─────────────────────────────────────────────
# CACHE STATS
# ─────────────────────────────────────────────
stats = embed_cache.stats

print(f"\n  \033[1;33m📊 Cache Statistics:\033[0m")

if isinstance(stats, dict):
    for k, v in stats.items():
        # Color hits green, misses red, others gray
        if "hit" in k.lower():
            val_color = "\033[1;32m"
        elif "miss" in k.lower():
            val_color = "\033[1;31m"
        else:
            val_color = "\033[1;37m"
        print(f"     \033[90m▸ {k:<18}\033[0m → {val_color}{v}\033[0m")
else:
    print(f"     \033[90m{stats}\033[0m")

# ─────────────────────────────────────────────
# INTERPRETATION
# ─────────────────────────────────────────────
print(f"\n  \033[1;32m💡 The second pass is dramatically faster —\033[0m")
print(f"  \033[1;32m   all from Redis, no API calls.\033[0m")

print("\n" + "═" * 68 + "\n")


════════════════════════════════════════════════════════════════════
  ⚡  SECOND PASS — CACHE-ONLY
  All embeddings should come from Redis (no API calls)
════════════════════════════════════════════════════════════════════

  ▸ Total time     → 24 ms
  ▸ Documents      → 15

  📊 Cache Statistics:
     ▸ hits               → 15
     ▸ misses             → 15
     ▸ hit_rate           → 50%
     ▸ keys_in_redis      → 15

  💡 The second pass is dramatically faster —
     all from Redis, no API calls.

════════════════════════════════════════════════════════════════════



## Retriever

In [5]:
import numpy as np

def cosine_similarity(a: list, b: list) -> float:
    a_arr = np.array(a)
    b_arr = np.array(b)
    return float(np.dot(a_arr, b_arr) / (np.linalg.norm(a_arr) * np.linalg.norm(b_arr)))


class CachedRetriever:
    """
    Retrieves relevant documents using cached embeddings.
    Also caches retrieval results so repeated queries are instant.
    """

    def __init__(self, redis_client, embed_cache: EmbeddingCache, documents: list):
        self.r = redis_client
        self.embed_cache = embed_cache
        self.documents = documents
        # Pre-load all document embeddings from cache
        self.doc_embeddings = {
            doc: embed_cache.get_or_compute(doc)
            for doc in documents
        }

    def retrieve(self, query: str, top_k: int = 3) -> list:
        """
        Find top_k most relevant documents for a query.
        Caches retrieval results for 30 minutes.
        """
        # Cache key for this exact query + k combination
        cache_key = f"retrieval:{hashlib.md5(f'{query}:{top_k}'.encode()).hexdigest()}"
        cached = self.r.get(cache_key)
        if cached:
            return json.loads(cached)

        # Compute query embedding (also benefits from embedding cache)
        query_embedding = self.embed_cache.get_or_compute(query)

        # Score all documents
        scored = []
        for doc, doc_emb in self.doc_embeddings.items():
            score = cosine_similarity(query_embedding, doc_emb)
            scored.append({"document": doc, "score": score})

        scored.sort(key=lambda x: x["score"], reverse=True)
        results = scored[:top_k]

        # Cache retrieval result for 30 minutes
        self.r.set(cache_key, json.dumps(results), ex=1800)

        return results


retriever = CachedRetriever(r, embed_cache, KNOWLEDGE_BASE)

# ─────────────────────────────────────────────
# PRETTY OUTPUT
# ─────────────────────────────────────────────
print("\n" + "═" * 68)
print("\033[1;36m  🔍  CACHED RETRIEVER — INITIALIZED\033[0m")
print("\033[90m  Two-layer cache: embeddings + retrieval results\033[0m")
print("═" * 68)

print(f"\n  \033[1;33m▸ Documents loaded\033[0m   → \033[1;37m{len(retriever.documents)}\033[0m")
print(f"  \033[1;33m▸ Embeddings\033[0m          → \033[1;32m✅ loaded from Redis\033[0m")
print(f"  \033[1;33m▸ Retrieval cache TTL\033[0m → \033[1;37m1800\033[0m sec  \033[90m(30 min)\033[0m")
print(f"  \033[1;33m▸ Default top_k\033[0m       → \033[1;37m3\033[0m")

print(f"\n  \033[1;32m✅ Available methods:\033[0m")
print(f"     \033[90m• retrieve(query, top_k=3)  → list of {{document, score}}\033[0m")

print(f"\n  \033[1;34m💡 Note:\033[0m \033[90mEach document embedding is computed ONCE and reused.\033[0m")
print(f"     \033[90mRepeated queries hit the retrieval cache (30 min TTL).\033[0m")

print("\n" + "═" * 68 + "\n")


════════════════════════════════════════════════════════════════════
  🔍  CACHED RETRIEVER — INITIALIZED
  Two-layer cache: embeddings + retrieval results
════════════════════════════════════════════════════════════════════

  ▸ Documents loaded   → 15
  ▸ Embeddings          → ✅ loaded from Redis
  ▸ Retrieval cache TTL → 1800 sec  (30 min)
  ▸ Default top_k       → 3

  ✅ Available methods:
     • retrieve(query, top_k=3)  → list of {document, score}

  💡 Note: Each document embedding is computed ONCE and reused.
     Repeated queries hit the retrieval cache (30 min TTL).

════════════════════════════════════════════════════════════════════



In [6]:
# Test the retriever
query = "What are the pricing plans?"

print("\n" + "═" * 74)
print("\033[1;36m  🔍  RETRIEVER TEST\033[0m")
print("═" * 74)

print(f"\n  \033[1;34m❓ Query:\033[0m \033[1;37m'{query}'\033[0m")

results = retriever.retrieve(query, top_k=3)

print(f"\n  \033[1;33m📄 Top {len(results)} results:\033[0m")
print("  " + "─" * 70)

for i, result in enumerate(results, 1):
    score = result['score']
    document = result['document']

    # Determine color based on similarity score
    if score >= 0.85:
        score_color = "\033[1;32m"      # green = very relevant
        label = "HIGH"
    elif score >= 0.70:
        score_color = "\033[1;36m"      # cyan = relevant
        label = "MED "
    elif score >= 0.50:
        score_color = "\033[1;33m"      # yellow = somewhat relevant
        label = "LOW "
    else:
        score_color = "\033[1;31m"      # red = not relevant
        label = "NONE"

    # Build a visual bar for the score (0-30 chars)
    bar_len = int(max(score, 0) * 30)
    bar = "█" * bar_len + "\033[90m" + "░" * (30 - bar_len) + "\033[0m"

    print(f"\n  \033[1;34m[{i}]\033[0m  {score_color}[{label}]\033[0m  "
          f"{score_color}{score:.3f}\033[0m  {bar}")
    print(f"       \033[90m{document}\033[0m")

print("\n" + "═" * 74 + "\n")


══════════════════════════════════════════════════════════════════════════
  🔍  RETRIEVER TEST
══════════════════════════════════════════════════════════════════════════

  ❓ Query: 'What are the pricing plans?'

  📄 Top 3 results:
  ──────────────────────────────────────────────────────────────────────

  [1]  [NONE]  0.459  █████████████░░░░░░░░░░░░░░░░░
       The Professional plan includes up to 75 users, 200GB storage, advanced analytics, and API access.

  [2]  [NONE]  0.449  █████████████░░░░░░░░░░░░░░░░░
       The Starter plan includes up to 10 users, 20GB storage, and basic reporting features.

  [3]  [NONE]  0.383  ███████████░░░░░░░░░░░░░░░░░░░
       InsightFlow offers three subscription tiers: Starter ($39/month), Professional ($119/month), and Enterprise (custom pricing).

══════════════════════════════════════════════════════════════════════════



## Answer Caching

In [8]:
from groq import Groq

groq_client = Groq()

class CachedRAG:
    """
    Full RAG pipeline with three layers of Redis caching:
    1. Embedding cache (permanent)
    2. Retrieval result cache (30 min)
    3. Final answer cache (1 hour)
    """

    def __init__(self, redis_client, retriever: CachedRetriever):
        self.r = redis_client
        self.retriever = retriever

    def _answer_cache_key(self, question: str) -> str:
        return f"rag:answer:{hashlib.md5(question.strip().lower().encode()).hexdigest()}"

    def answer(self, question: str, top_k: int = 3) -> dict:
        start = time.time()

        # Check answer cache first
        cache_key = self._answer_cache_key(question)
        cached_answer = self.r.get(cache_key)
        if cached_answer:
            data = json.loads(cached_answer)
            data["source"] = "ANSWER CACHE HIT"
            data["time_ms"] = round((time.time() - start) * 1000, 2)
            return data

        # Retrieve relevant documents
        retrieved = self.retriever.retrieve(question, top_k=top_k)
        context_parts = [f"Document {i+1}: {r['document']}" for i, r in enumerate(retrieved)]
        context = "\n".join(context_parts)

        # Build prompt for LLM
        prompt = f"""You are a helpful customer support assistant for TechCorp.
Use ONLY the information in the provided documents to answer the question.
If the documents don't contain enough information to answer, say so.

Documents:
{context}

Question: {question}

Answer:"""

        # Generate answer using Groq
        response = groq_client.chat.completions.create(
            model="openai/gpt-oss-120b",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=300
        )

        answer = response.choices[0].message.content
        elapsed_ms = round((time.time() - start) * 1000, 2)

        result = {
            "question": question,
            "answer": answer,
            "sources": [r["document"][:100] + "..." for r in retrieved],
            "source": "COMPUTED",
            "time_ms": elapsed_ms
        }

        # Cache the final answer for 1 hour
        self.r.set(cache_key, json.dumps(result), ex=3600)

        return result


rag = CachedRAG(r, retriever)

# ─────────────────────────────────────────────
# PRETTY OUTPUT
# ─────────────────────────────────────────────
print("\n" + "═" * 72)
print("\033[1;36m  🧠  CACHED RAG PIPELINE — READY\033[0m")
print("\033[90m  Three-layer Redis caching for cost & latency optimization\033[0m")
print("═" * 72)

print(f"\n  \033[1;33m🏗️   Cache Layers:\033[0m")
print(f"     \033[1;34m[1]\033[0m \033[1;37mEmbedding cache\033[0m     → \033[1;32mPERMANENT\033[0m  \033[90m(per document/query)\033[0m")
print(f"     \033[1;34m[2]\033[0m \033[1;37mRetrieval cache\033[0m     → \033[1;33m30 min\033[0m     \033[90m(query + top_k → docs)\033[0m")
print(f"     \033[1;34m[3]\033[0m \033[1;37mAnswer cache\033[0m        → \033[1;33m60 min\033[0m     \033[90m(final LLM response)\033[0m")

print(f"\n  \033[1;33m⚙️   Configuration:\033[0m")
print(f"     \033[90m▸ LLM model       : \033[1;37mopenai/gpt-oss-120b\033[0m \033[90m(via Groq)\033[0m")
print(f"     \033[90m▸ Answer cache key: \033[1;37mrag:answer:<md5>\033[0m")
print(f"     \033[90m▸ Default top_k   : \033[1;37m3\033[0m")

print(f"\n  \033[1;32m✅ Available methods:\033[0m")
print(f"     \033[90m• answer(question, top_k=3)  → dict with question/answer/sources/source/time_ms\033[0m")

print(f"\n  \033[1;35m⚡ Expected behavior:\033[0m")
print(f"     \033[90mFirst call  → \033[1;33mCOMPUTED\033[0m \033[90m(full pipeline: embed + retrieve + LLM)\033[0m")
print(f"     \033[90mRepeat call → \033[1;32mANSWER CACHE HIT\033[0m \033[90m(instant, ~0 ms)\033[0m")

print("\n" + "═" * 72 + "\n")


════════════════════════════════════════════════════════════════════════
  🧠  CACHED RAG PIPELINE — READY
  Three-layer Redis caching for cost & latency optimization
════════════════════════════════════════════════════════════════════════

  🏗️   Cache Layers:
     [1] Embedding cache     → PERMANENT  (per document/query)
     [2] Retrieval cache     → 30 min     (query + top_k → docs)
     [3] Answer cache        → 60 min     (final LLM response)

  ⚙️   Configuration:
     ▸ LLM model       : openai/gpt-oss-120b (via Groq)
     ▸ Answer cache key: rag:answer:<md5>
     ▸ Default top_k   : 3

  ✅ Available methods:
     • answer(question, top_k=3)  → dict with question/answer/sources/source/time_ms

  ⚡ Expected behavior:
     First call  → COMPUTED (full pipeline: embed + retrieve + LLM)
     Repeat call → ANSWER CACHE HIT (instant, ~0 ms)

════════════════════════════════════════════════════════════════════════



In [9]:
# Clear any cached answers for a clean demo
for k in r.keys("rag:answer:*"):
    r.delete(k)

# Ask a question for the first time
question = "What is the refund policy?"

print("\n" + "═" * 66)
print("\033[1;36m  🌐  FIRST CALL\033[0m  \033[90m— computing from scratch (expected: COMPUTED)\033[0m")
print("═" * 66)

result = rag.answer(question)

# Determine color, icon and label based on source
source = result['source']
if "ANSWER CACHE HIT" in source:
    source_display = "\033[1;32m⚡ ANSWER CACHE HIT\033[0m"
elif "COMPUTED" in source:
    source_display = "\033[1;33m🌐 COMPUTED\033[0m"
else:
    source_display = f"\033[1;37m{source}\033[0m"

print(f"\n  \033[1;34m▸ Source\033[0m     → {source_display}")
print(f"  \033[1;34m▸ Time\033[0m       → \033[1;32m{result['time_ms']} ms\033[0m")

print(f"\n  \033[1;34m▸ Answer\033[0m")
print(f"     \033[90m{result['answer']}\033[0m")

print(f"\n  \033[1;34m▸ Source documents used:\033[0m")
for i, src in enumerate(result['sources'], 1):
    print(f"     \033[90m[{i}]\033[0m \033[1;37m{src}\033[0m")

print("\n" + "═" * 66 + "\n")


══════════════════════════════════════════════════════════════════
  🌐  FIRST CALL  — computing from scratch (expected: COMPUTED)
══════════════════════════════════════════════════════════════════

  ▸ Source     → 🌐 COMPUTED
  ▸ Time       → 2244.18 ms

  ▸ Answer
     NovaTech’s refund policy is as follows:

- **Starter and Professional plans:** 21‑day money‑back guarantee.  
- **Enterprise contracts:** Non‑refundable after 45 days.

  ▸ Source documents used:
     [1] NovaTech's refund policy: Starter and Professional plans have a 21-day money-back guarantee. Enterpr...
     [2] NovaTech's customer support operates 24/7 for Enterprise customers and 8am-8pm CST Monday-Saturday f...
     [3] Enterprise customers get unlimited users, dedicated infrastructure, SLA guarantees, and a dedicated ...

══════════════════════════════════════════════════════════════════



In [11]:
# Run several questions to see all three cache layers working
questions = [
    "What integrations does DataMind support?",
    "Is NovaTech GDPR compliant?",
    "How many users can the Professional plan support?",
    "What integrations does DataMind support?",   # repeated — answer cache hit
    "Is NovaTech GDPR compliant?",               # repeated — answer cache hit
    "Where is NovaTech data stored?",
    "What is the Enterprise plan price?",
]

print("\n" + "═" * 82)
print("\033[1;36m  🧪  MULTI-QUESTION CACHE TEST\033[0m")
print(f"\033[90m  {len(questions)} questions · duplicates should trigger cache hits\033[0m")
print("═" * 82)

# ─────────────────────────────────────────────
# TABLE HEADER
# ─────────────────────────────────────────────
print(f"\n  \033[1;33m{'#':>2}  {'SOURCE':<20} {'TIME (ms)':>10}  QUESTION\033[0m")
print("  " + "─" * 78)

# ─────────────────────────────────────────────
# PROCESS EACH QUESTION
# ─────────────────────────────────────────────
cache_hits = 0
computed = 0

for i, q in enumerate(questions, 1):
    result = rag.answer(q)

    # Determine color, icon and label based on source
    if "CACHE" in result["source"]:
        source_label = "\033[1;32m⚡ CACHE HIT\033[0m"
        cache_hits += 1
    else:
        source_label = "\033[1;33m🌐 COMPUTED\033[0m"
        computed += 1

    # Color the time based on whether it was cached or not
    time_color = "\033[1;32m" if "CACHE" in result["source"] else "\033[1;33m"

    print(f"  \033[90m{i:>2}\033[0m  {source_label:<22} "
          f"{time_color}{result['time_ms']:>10.1f}\033[0m  "
          f"\033[90m{q[:48]}\033[0m")

print("  " + "─" * 78)

# ─────────────────────────────────────────────
# SUMMARY
# ─────────────────────────────────────────────
total = len(questions)
hit_rate = cache_hits / total * 100

print(f"\n  \033[1;34m📊 Summary:\033[0m")
print(f"     \033[1;33m🌐 Computed\033[0m    → \033[1;33m{computed:>2}\033[0m")
print(f"     \033[1;32m⚡ Cache hits\033[0m   → \033[1;32m{cache_hits:>2}\033[0m")
print(f"     \033[1;34m📈 Hit rate\033[0m     → \033[1;37m{hit_rate:.0f}%\033[0m")

print(f"\n  \033[1;32m💡 Notice the cached questions are significantly faster!\033[0m")

print("\n" + "═" * 82 + "\n")


══════════════════════════════════════════════════════════════════════════════════
  🧪  MULTI-QUESTION CACHE TEST
  7 questions · duplicates should trigger cache hits
══════════════════════════════════════════════════════════════════════════════════

   #  SOURCE                TIME (ms)  QUESTION
  ──────────────────────────────────────────────────────────────────────────────
   1  ⚡ CACHE HIT        2.9  What integrations does DataMind support?
   2  🌐 COMPUTED       608.8  Is NovaTech GDPR compliant?
   3  ⚡ CACHE HIT        1.5  How many users can the Professional plan support
   4  ⚡ CACHE HIT        1.1  What integrations does DataMind support?
   5  ⚡ CACHE HIT        1.0  Is NovaTech GDPR compliant?
   6  🌐 COMPUTED       577.2  Where is NovaTech data stored?
   7  ⚡ CACHE HIT        4.2  What is the Enterprise plan price?
  ──────────────────────────────────────────────────────────────────────────────

  📊 Summary:
     🌐 Computed    →  2
     ⚡ Cache hits   →  5
     📈 Hit r